In [1]:
pip install pandas


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install numpy



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import numpy as np

# --------------------------------------------------
# 1. Load the data
# --------------------------------------------------
df = pd.read_csv("C:\\Users\\morte\\Downloads\\FPMA-main\\wheat\\wheat.csv")

# --------------------------------------------------
# 2. Parse date correctly
# --------------------------------------------------
df["date"] = pd.to_datetime(df["date"], dayfirst=True)

df.head()


,date,price_usd,commodity_name,commodity_type,country,iso3,region,subregion,market,price_type,unit,unit_std,price_per_unit,price_source,gdp_ppp17,gdppc_ppp,gnipc_ppp,population,inflation,income_level
0,2000-01-01,0.29,Bread,Bread,Afghanistan,AFG,Asia,Southern Asia,Herat,Retail,Kg,kg,0.29,Domestic,NaN,NaN,NaN,42647492.0,-6.601186,NaN
1,2000-02-01,0.29,Bread,Bread,Afghanistan,AFG,Asia,Southern Asia,Herat,Retail,Kg,kg,0.29,Domestic,NaN,NaN,NaN,42647492.0,-6.601186,NaN
2,2000-03-01,0.29,Bread,Bread,Afghanistan,AFG,Asia,Southern Asia,Herat,Retail,Kg,kg,0.29,Domestic,NaN,NaN,NaN,42647492.0,-6.601186,NaN
3,2000-04-01,0.28,Bread,Bread,Afghanistan,AFG,Asia,Southern Asia,Herat,Retail,Kg,kg,0.28,Domestic,NaN,NaN,NaN,42647492.0,-6.601186,NaN
4,2000-05-01,0.27,Bread,Bread,Afghanistan,AFG,Asia,Southern Asia,Herat,Retail,Kg,kg,0.27,Domestic,NaN,NaN,NaN,42647492.0,-6.601186,NaN


In [4]:
# Define series identity
group_cols = [
    "commodity_name",
    "commodity_type",
    "price_source",
    "country",
    "market",
    "price_type",
    "unit_std"
]

# Sort correctly
df = df.sort_values(group_cols + ["date"])


In [5]:
def compute_gap_lengths(s):
    is_na = s.isna()
    gap_id = (is_na != is_na.shift()).cumsum()

    return (
        s[is_na]
        .groupby(gap_id)
        .size()
    )

gap_report = (
    df
    .groupby(group_cols)["price_per_unit"]
    .apply(compute_gap_lengths)
    .reset_index(name="gap_length_months")
)


In [6]:
gap_report.head(10)


,commodity_name,commodity_type,price_source,country,market,price_type,unit_std,level_7,gap_length_months
0,Bread,Bread,Domestic,Afghanistan,Kabul,Retail,kg,2,1
1,Bread,Bread,Domestic,Afghanistan,Kandahar,Retail,kg,2,3
2,Bread,Bread,Domestic,Algeria,Awserd,Retail,unit,2,3
3,Bread,Bread,Domestic,Algeria,Awserd,Retail,unit,4,64
4,Bread,Bread,Domestic,Algeria,Awserd,Retail,unit,6,1
5,Bread,Bread,Domestic,Algeria,Awserd,Retail,unit,8,1
6,Bread,Bread,Domestic,Algeria,Awserd,Retail,unit,10,1
7,Bread,Bread,Domestic,Algeria,Awserd,Retail,unit,12,11
8,Bread,Bread,Domestic,Algeria,Awserd,Retail,unit,14,1
9,Bread,Bread,Domestic,Algeria,Awserd,Retail,unit,16,1


In [10]:
gap_report.describe()


,level_7,gap_length_months
count,852.000000,852.000000
mean,10.666667,3.625587
std,12.926863,7.639157
min,2.000000,1.000000
25%,2.000000,1.000000
50%,6.000000,1.000000
75%,12.000000,4.000000
max,84.000000,128.000000


In [11]:
gap_report["gap_length_months"].value_counts().sort_index()


gap_length_months
1      447
2      132
3       57
4       55
5       26
6       19
7       13
8       19
9       27
10      11
11      15
12       7
13       3
14       2
17       1
19       1
22       1
23       2
26       1
28       2
38       1
44       2
50       4
63       1
64       2
128      1
Name: count, dtype: int64

In [6]:
for m in [2, 3, 4, 6]:
    pct = (gap_report["gap_length_months"] <= m).mean() * 100
    print(f"≤ {m} months: {pct:.2f}% of gaps")


≤ 2 months: 67.96% of gaps
≤ 3 months: 74.65% of gaps
≤ 4 months: 81.10% of gaps
≤ 6 months: 86.38% of gaps


In [8]:
(gap_report["gap_length_months"] <= 4).mean() * 100


np.float64(81.10328638497653)

In [7]:
import pandas as pd

raw = pd.read_csv("C:\\Users\\morte\\Downloads\\FPMA-main\\wheat\\wheat.csv")

before = raw["price_per_unit"].isna().sum()
print("Missing before:", before)


Missing before: 3089


In [12]:
def spline_fill_short_gaps(s, max_gap=4):
    is_na = s.isna()
    
    # Identify NA runs
    na_runs = (is_na & ~is_na.shift(fill_value=False)).cumsum()
    gap_size = is_na.groupby(na_runs).transform("sum")
    
    # Determine which NaN values are in short gaps
    allowed = is_na & (gap_size <= max_gap)
    
    # Interpolate using ALL original data (including valid points)
    out = s.copy()
    if s.notna().sum() >= 4:
        interpolated = s.interpolate(method="spline", order=3)
        # Only fill where allowed
        out[allowed] = interpolated[allowed]
    
    return out

df["price_per_unit_filled"] = (
    df
    .groupby(group_cols)["price_per_unit"]
    .transform(lambda s: spline_fill_short_gaps(s, max_gap=6))
)

df["price_per_unit"] = df["price_per_unit_filled"]
df = df.drop(columns="price_per_unit_filled")

df.to_csv(
    "C:\\Users\\morte\\Downloads\\FPMA-main\\wheat\\wheat_interpolated.csv",
    index=False
)


In [11]:
print(set(df.columns))


{'region', 'commodity_type', 'unit_std', 'gdp_ppp17', 'date', 'commodity_name', 'country', 'gnipc_ppp', 'unit', 'income_level', 'price_source', 'price_type', 'population', 'price_usd', 'iso3', 'market', 'gdppc_ppp', 'subregion', 'price_per_unit', 'inflation'}


In [11]:
after = df["price_per_unit"].isna().sum()
print("Missing after:", after)

filled = before - after
print("Filled values:", filled)


Missing after: 1987
Filled values: 1102


In [ ]:
# See what 6 months would give you
for max_gap in [2, 3, 4, 6, 12]:
    df_test = df.copy()
    df_test["price_per_unit"] = (
        df_test
        .groupby(group_cols)["price_per_unit"]
        .transform(lambda s: spline_fill_short_gaps(s, max_gap=max_gap))
    )
    filled = before - df_test["price_per_unit"].isna().sum()
    print(f"max_gap={max_gap:2d}: {filled:4d} filled ({filled/before*100:.1f}%)")


max_gap= 2: 1102 filled (35.7%)
max_gap= 3: 1102 filled (35.7%)
max_gap= 4: 1102 filled (35.7%)
max_gap= 6: 1346 filled (43.6%)
max_gap=12: 2191 filled (70.9%)


In [ ]:
# Verify what you're actually filling with max_gap=6
gap_breakdown = gap_report[gap_report["gap_length_months"] <= 6]
print(f"With max_gap=6, you're filling gaps of lengths:")
print(gap_breakdown["gap_length_months"].value_counts().sort_index())


With max_gap=6, you're filling gaps of lengths:
gap_length_months
1    447
2    132
3     57
4     55
5     26
6     19
Name: count, dtype: int64
